> **Notebook 1 of 4 - Cleaning**  
> Input: `data/Online Retail.xlsx` · Output: `data/sales_clean.csv`  
> Next: `02_eda.ipynb`

# Online Retail Customer Analytics & Segmentation

# 1. Business Understanding

This project analyzes one year of transactions from a UK-based online retailer (Dec 2010 – Dec 2011). The goal is to turn raw transaction lines into customer-level insight and actionable customer segments for marketing and retention.

**Key business questions**

- Which products generate the most units and revenue?
- Which customers contribute the most revenue and orders?
- How does revenue change over time (months, weekdays, hours, seasonality)?
- Which countries generate the most customers, orders, and revenue?
- How common are cancellations/returns, and which products/customers are linked to them?
- What does a typical customer look like, and is there wholesale/bulk-buying behavior?
- Can customers be grouped into actionable segments based on value and engagement?

# 2. Dataset Loading & Documentation

The dataset is the public *Online Retail* dataset (`Online Retail.xlsx`). Each row is one **transaction line** (one product on one invoice), not one order.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import sys

sys.path.append('../src')
from cleaning import clean_data, add_features

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

In [2]:
df = pd.read_excel("../data/Online Retail.xlsx")
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [3]:
df.shape

(541909, 8)

| Column | Meaning |
|---|---|
| `InvoiceNo` | Invoice number. Starts with `C` for cancellations/returns |
| `StockCode` | Product code |
| `Description` | Product name |
| `Quantity` | Units on the line (negative for returns) |
| `InvoiceDate` | Date and time of the invoice |
| `UnitPrice` | Price per unit (£) |
| `CustomerID` | Customer identifier (a label, not a numeric feature) |
| `Country` | Customer country |

# 3. Data Audit

Before changing anything, we check structure, missing values, and suspicious records.

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 33.1+ MB


In [5]:
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [6]:
df.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID
count,541909.000000,541909,541909.000000,406829.000000
mean,9.552250,2011-07-04 13:34:57.156386,4.611114,15287.690570
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:34:00,1.250000,13953.000000
50%,3.000000,2011-07-19 17:17:00,2.080000,15152.000000
75%,10.000000,2011-10-19 11:27:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,218.081158,NaN,96.759853,1713.600303


In [7]:
is_cancel = df['InvoiceNo'].astype(str).str.startswith('C')

print("Exact duplicate lines :", df.duplicated().sum())
print("Cancellation lines    :", is_cancel.sum())
print("UnitPrice == 0        :", (df['UnitPrice'] == 0).sum())
print("UnitPrice < 0         :", (df['UnitPrice'] < 0).sum())
print("Missing CustomerID    :", df['CustomerID'].isna().sum())

Exact duplicate lines : 5268
Cancellation lines    : 9288
UnitPrice == 0        : 2515
UnitPrice < 0         : 2
Missing CustomerID    : 135080


**Audit findings**

- 541,909 lines, 25,900 invoices, 4,372 customer IDs, 38 countries.
- 5,268 exact duplicate lines.
- 9,288 cancellation lines (`InvoiceNo` starts with `C`, negative quantity).
- 2,515 lines with `UnitPrice = 0` and 2 lines with a negative price.
- 135,080 lines (about 25%) have no `CustomerID`.
- `InvoiceDate` is already a datetime column.

# 4. Cleaning & Data Quality Decisions

Each decision below is documented together with its business impact.

### 4.1 Cancellations and returns

In [8]:
# Are cancellation invoices linked to an original invoice number?
cancel_invoices = df.loc[is_cancel, 'InvoiceNo'].astype(str).str[1:]
original_invoices = set(df.loc[~is_cancel, 'InvoiceNo'].astype(str))
print("Cancellations matching an original InvoiceNo:", cancel_invoices.isin(original_invoices).sum())

Cancellations matching an original InvoiceNo: 0


In [9]:
# Second attempt: match on Customer + Product + Quantity
known = df[df['CustomerID'].notna()]
purchases_keys = set(zip(known.loc[~is_cancel, 'CustomerID'],
                         known.loc[~is_cancel, 'StockCode'],
                         known.loc[~is_cancel, 'Quantity']))

cancel_known = known[known['InvoiceNo'].astype(str).str.startswith('C')]
cancel_keys = zip(cancel_known['CustomerID'],
                  cancel_known['StockCode'],
                  -cancel_known['Quantity'])

matched = pd.Series([k in purchases_keys for k in cancel_keys])
print("Cancellation lines with a known customer:", len(matched))
print("Matched to a purchase of the same customer/product/quantity: %.1f%%" % (matched.mean() * 100))

Cancellation lines with a known customer: 8905
Matched to a purchase of the same customer/product/quantity: 36.7%


No cancellation invoice matches an original invoice number (a `C` invoice has its own number). Matching on customer + product + quantity finds an original purchase for only **36.7%** of the 8,905 cancellation lines with a known customer, so exact netting is not reliable.

**Decision:** keep cancellations. Their negative quantities produce negative revenue, so returns reduce net revenue automatically. Some cancellations are full reversals of a purchase made minutes earlier (see the basket section).

### 4.2 Zero and negative prices

In [10]:
df[df['UnitPrice'] < 0][['InvoiceNo', 'Description', 'Quantity', 'UnitPrice']]

,InvoiceNo,Description,Quantity,UnitPrice
299983,A563186,Adjust bad debt,1,-11062.06
299984,A563187,Adjust bad debt,1,-11062.06


In [11]:
df[df['UnitPrice'] == 0]['Description'].value_counts(dropna=False).head(8)

Description
NaN                      1454
check                     159
?                          47
damages                    45
damaged                    43
found                      25
sold as set on dotcom      20
adjustment                 16
Name: count, dtype: int64

The two negative prices are `Adjust bad debt` accounting entries. The zero-price lines are mostly missing descriptions, `check`, `damages`, `found`, and `adjustment`, which are stock adjustments and not sales.

**Decision:** exclude all lines with `UnitPrice <= 0` from revenue analysis. Keeping them would add fake quantity with no revenue and distort product analysis.

### 4.3 Missing CustomerID

In [12]:
# Guest-checkout check: is CustomerID missing for a whole invoice or only for some lines?
missing_share = df.groupby('InvoiceNo')['CustomerID'].apply(lambda s: s.isna().mean())

print("Invoices with ALL lines missing CustomerID :", (missing_share == 1).sum())
print("Invoices with SOME lines missing CustomerID:", ((missing_share > 0) & (missing_share < 1)).sum())

Invoices with ALL lines missing CustomerID : 3710
Invoices with SOME lines missing CustomerID: 0


All **3,710** invoices without a `CustomerID` are missing it on **every** line (none are partial), so they behave like guest-checkout blocks.

**Decision:** keep them for transaction-level analysis (sales, products, time, geography) because they are real revenue, and exclude them only from customer-level work (RFM and clustering) because they cannot be linked to a customer.

### 4.4 Exact duplicates

In [13]:
# Steps implemented in src/cleaning.py: remove exact duplicates, keep UnitPrice > 0
print("After removing duplicates:", df.drop_duplicates().shape)

sales_df = clean_data(df)
print("Analysis dataset (UnitPrice > 0):", sales_df.shape)

After removing duplicates: (536641, 8)


Analysis dataset (UnitPrice > 0): (534129, 8)


### Cleaning summary

| Issue | Decision | Business impact |
|---|---|---|
| 5,268 exact duplicate lines | Removed | Avoids double-counting sales |
| Cancellations / returns | Kept as negative revenue | Revenue is net of returns |
| `UnitPrice <= 0` | Excluded from `sales_df` | Removes adjustments and non-sales |
| Missing `CustomerID` | Kept, excluded only from customer-level analysis | Keeps total revenue correct |

The analysis dataset `sales_df` has **534,129 lines**.

# 5. Feature Engineering

The features below are created by `add_features()` in `src/cleaning.py`:

- `TotalLineRevenue = Quantity × UnitPrice` (negative for returns).
- `Year`, `Month`, `DayOfWeek`, `Hour` extracted from `InvoiceDate` for time patterns.
- `Is_UK` flag to separate the UK market from international sales.

In [14]:
# TotalLineRevenue, Year, Month, DayOfWeek, Hour and Is_UK are created in src/cleaning.py
sales_df = add_features(sales_df)

sales_df[['InvoiceDate', 'Quantity', 'UnitPrice', 'TotalLineRevenue', 'Year', 'Month', 'DayOfWeek', 'Hour', 'Is_UK']].head()

,InvoiceDate,Quantity,UnitPrice,TotalLineRevenue,Year,Month,DayOfWeek,Hour,Is_UK
0,2010-12-01 08:26:00,6,2.55,15.30,2010,12,2,8,1
1,2010-12-01 08:26:00,6,3.39,20.34,2010,12,2,8,1
2,2010-12-01 08:26:00,8,2.75,22.00,2010,12,2,8,1
3,2010-12-01 08:26:00,6,3.39,20.34,2010,12,2,8,1
4,2010-12-01 08:26:00,6,3.39,20.34,2010,12,2,8,1


In [15]:
print(sales_df.shape)
print("Invoices:", sales_df['InvoiceNo'].nunique(), "| Products:", sales_df['StockCode'].nunique(),
      "| Customers:", sales_df['CustomerID'].nunique(), "| Countries:", sales_df['Country'].nunique())
print("Period:", sales_df['InvoiceDate'].min(), "->", sales_df['InvoiceDate'].max())

(534129, 14)


Invoices: 23796 | Products: 3938 | Customers: 4371 | Countries: 38
Period: 2010-12-01 08:26:00 -> 2011-12-09 12:50:00


The analysis dataset covers **23,796 invoices, 3,938 products, 4,371 customers and 38 countries** between 1 Dec 2010 and 9 Dec 2011. Very large quantities are kept for now because they may be genuine wholesale orders; they are examined in the EDA.

In [16]:
# Save the cleaned data for the next notebooks
sales_df.to_csv('../data/sales_clean.csv', index=False)
print("Saved:", sales_df.shape)

Saved: (534129, 14)
